# Crawler hsctvn.com — 7 trường dữ liệu
**Dự án E14 | MCNA Onsite 2025**

| | |
|---|---|
| Nguồn listing | `hsctvn.com/p/ha-noi` & `hsctvn.com/p/ho-chi-minh` |
| Nguồn detail | `hsctvn.com/{href}.htm` |
| Chống Cloudflare | `playwright` — giả lập Chrome thật |
| Output | `data/doanh_nghiep.csv` + `data/doanh_nghiep.xlsx` |

**7 trường:** MST · Tên CT · Năm TL · SĐT · Email · Địa chỉ · Ngành nghề


## Bước 1 — Cài thư viện

In [ ]:
%pip install playwright openpyxl pandas unidecode beautifulsoup4 -q
import subprocess
subprocess.run(["python", "-m", "playwright", "install", "chromium"], check=True)
print("Done!")

## Bước 2 — Import & Config

In [ ]:
import re, os, time, random, logging
from datetime import datetime
from unidecode import unidecode
import pandas as pd
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("crawler_hsctvn.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

BASE_URL = "https://hsctvn.com"
TINH_URL = {
    "ha-noi"     : "/p/ha-noi",
    "ho-chi-minh": "/p/ho-chi-minh",
}
DELAY_MIN  = 2.0
DELAY_MAX  = 4.0
os.makedirs("data", exist_ok=True)
CSV_PATH   = "data/doanh_nghiep.csv"
EXCEL_PATH = "data/doanh_nghiep.xlsx"
CKPT_PATH  = "data/checkpoint.csv"

print("Config xong!")
print(f"  HN : {BASE_URL}{TINH_URL['ha-noi']}")
print(f"  HCM: {BASE_URL}{TINH_URL['ho-chi-minh']}")


## Bước 3 — Hàm tiện ích

In [ ]:
def normalize_phone(phone):
    if not phone: return ""
    digits = re.sub(r"\D", "", phone)
    if digits.startswith("84") and len(digits) == 11:
        digits = "0" + digits[2:]
    return digits if len(digits) in (9,10) and digits.startswith("0") else phone.strip()

def normalize_address(addr):
    if not addr: return ""
    return " ".join(addr.split()).replace(" ,", ",")

def decode_cf_email(encoded):
    try:
        enc = bytes.fromhex(encoded)
        key = enc[0]
        return "".join(chr(b ^ key) for b in enc[1:]).lower().strip()
    except Exception:
        return ""

def build_detail_url(href):
    return f"{BASE_URL}/{href.lstrip('/')}"

print("Ham tien ich san sang!")
print(f"  Test decode email: {decode_cf_email('4a3e382324222925242d2e3f33737f0a2d272b232664292527')}")


## Bước 4 — Khoi dong Playwright Browser
> `headless=True` = an cua so Chrome (nhanh hon)
> `headless=False` = hien cua so (de debug)


In [ ]:
_pw     = sync_playwright().start()
browser = _pw.chromium.launch(
    headless=True,
    args=["--no-sandbox", "--disable-blink-features=AutomationControlled"]
)
context = browser.new_context(
    user_agent=(
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    viewport={"width": 1280, "height": 800},
    locale="vi-VN",
)
page = context.new_page()
page.add_init_script("Object.defineProperty(navigator,'webdriver',{get:()=>undefined})")

print("Browser Chromium da khoi dong!")


## Bước 5 — Ham fetch HTML

In [ ]:
def fetch_html(url, retries=3):
    for attempt in range(1, retries + 1):
        try:
            page.goto(url, wait_until="domcontentloaded", timeout=20000)
            page.wait_for_timeout(random.randint(800, 1500))
            html = page.content()
            if len(html) > 500:
                return html
            log.warning(f"HTML qua ngan ({len(html)} chars)")
            time.sleep(5)
        except Exception as e:
            log.error(f"Loi lan {attempt}: {e}")
            time.sleep(5 * attempt)
    return None

# Test fetch
print("Test fetch trang Ha Noi...")
html_test = fetch_html(f"{BASE_URL}{TINH_URL['ha-noi']}")
if html_test:
    print(f"OK! HTML dai {len(html_test):,} ky tu")
    print(f"100 ky tu dau: {html_test[:100]}")
else:
    print("THAT BAI — kiem tra ket noi mang")


## Bước 6 — Parser trang listing
> Cau truc HTML that:
> `<ul class="hsdn"><li><h3><a href="cong-ty-xxx-com-ID.htm" title="MST - TEN">`
> MST lay tu `title` attribute
> href dung de build URL chi tiet


In [ ]:
def parse_listing(html, tinh_label):
    soup    = BeautifulSoup(html, "html.parser")
    records = []
    seen    = set()

    listing = soup.find("ul", class_="hsdn")
    if not listing:
        log.warning("Khong tim thay ul.hsdn")
        return []

    for li in listing.find_all("li", recursive=False):
        h3 = li.find("h3")
        if not h3: continue
        a = h3.find("a", href=True)
        if not a: continue

        href  = a.get("href", "")
        title = a.get("title", "")   # dang "0111482863 - TEN CONG TY"
        ten   = a.get_text(strip=True)

        # Lay MST tu title
        mst = title.split(" - ")[0].strip() if " - " in title else ""
        if not mst or len(mst) < 10 or mst in seen:
            continue
        seen.add(mst)

        # Dia chi trong <div>
        div_tag = li.find("div")
        dia_chi = ""
        if div_tag:
            raw = div_tag.get_text(separator=" ", strip=True)
            raw = re.sub(r"Ma so thue:.*$", "", raw)
            raw = re.sub(r"^[Dd]ia chi[: ]*", "", raw)
            dia_chi = normalize_address(raw)

        records.append({
            "ma_so_thue"   : mst,
            "ten_cong_ty"  : ten,
            "dia_chi"      : dia_chi,
            "tinh_thanh"   : tinh_label,
            "detail_href"  : href,
            "nam_thanh_lap": "",
            "so_dien_thoai": "",
            "email"        : "",
            "nganh_nghe"   : "",
            "crawled_at"   : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })

    return records

# Test
if html_test:
    recs = parse_listing(html_test, "Ha Noi")
    print(f"Trang 1 Ha Noi: {len(recs)} records")
    for r in recs[:3]:
        print(f"  {r['ma_so_thue']} | {r['ten_cong_ty'][:40]}")
        print(f"  Dia chi: {r['dia_chi'][:60]}")
        print(f"  Href   : {r['detail_href']}")
        print()


## Bước 7 — Parser trang chi tiet
> Cau truc: `<ul class='hsct'>` chua SDT, Email (CF encoded), Nam TL
> Email bi Cloudflare ma hoa → dung `decode_cf_email()`


In [ ]:
def parse_detail(html):
    soup   = BeautifulSoup(html, "html.parser")
    result = {"so_dien_thoai": "", "email": "", "nam_thanh_lap": "", "nganh_nghe": ""}

    for ul in soup.find_all("ul", class_="hsct"):
        for li in ul.find_all("li"):
            li_str = str(li)
            text   = li.get_text(separator=" ", strip=True)

            # SDT
            if "fa-phone" in li_str or "dien thoai" in text.lower():
                nums = re.findall(r"0\d{8,9}", text)
                if nums and not result["so_dien_thoai"]:
                    result["so_dien_thoai"] = normalize_phone(nums[0])

            # Email — decode Cloudflare
            elif "fa-envelope" in li_str or "email" in text.lower():
                cf = li.find(class_="__cf_email__")
                if cf and cf.get("data-cfemail"):
                    result["email"] = decode_cf_email(cf["data-cfemail"])
                else:
                    emails = re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", text)
                    if emails:
                        result["email"] = emails[0].lower()

            # Nam thanh lap (tu Ngay cap)
            elif "fa-calendar" in li_str or "ngay cap" in text.lower():
                years = re.findall(r"\b(19|20)\d{2}\b", text)
                if years and not result["nam_thanh_lap"]:
                    result["nam_thanh_lap"] = years[0]

    return result

# Test voi cong ty SOMA (HCM)
print("Test parse chi tiet...")
test_detail_url = f"{BASE_URL}/cong-ty-tnhh-mtv-cham-soc-suc-khoe-toan-dien-soma-com-3794602.htm"
detail_html     = fetch_html(test_detail_url)
if detail_html:
    r = parse_detail(detail_html)
    print("Parse OK!")
    print(f"  SDT          : {r['so_dien_thoai']}")
    print(f"  Email        : {r['email']}")
    print(f"  Nam TL       : {r['nam_thanh_lap']}")
    print(f"  Nganh nghe   : {r['nganh_nghe'] or '(khong co tren site)'}")


## Bước 8 — Ham crawl listing theo tinh

In [ ]:
def get_listing_url(tinh_key, page_num):
    base = TINH_URL[tinh_key]
    if page_num == 1:
        return f"{BASE_URL}{base}"
    return f"{BASE_URL}{base}/page-{page_num}"

def crawl_listing(tinh_key, tinh_label, max_records=500):
    all_records = []
    seen_mst    = set()
    page_num    = 1
    empty_count = 0

    log.info(f"Crawl listing: {tinh_label} | muc tieu: {max_records}")

    while len(all_records) < max_records:
        url  = get_listing_url(tinh_key, page_num)
        html = fetch_html(url)

        if not html:
            log.error(f"Khong fetch duoc trang {page_num}")
            break

        records   = parse_listing(html, tinh_label)
        new_count = 0

        for r in records:
            if r["ma_so_thue"] not in seen_mst and len(all_records) < max_records:
                seen_mst.add(r["ma_so_thue"])
                all_records.append(r)
                new_count += 1

        log.info(f"  [{tinh_label}] Trang {page_num:>4}: +{new_count:>2} | Tong: {len(all_records):>5}/{max_records}")

        if new_count == 0:
            empty_count += 1
            if empty_count >= 2:
                log.info("  2 trang trong → dung")
                break
        else:
            empty_count = 0

        page_num += 1

        if page_num % 5 == 0:
            wait = random.uniform(6, 10)
            log.info(f"  Nghi {wait:.1f}s...")
            time.sleep(wait)
        else:
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    log.info(f"Xong listing {tinh_label}: {len(all_records)} records")
    return all_records

print("crawl_listing() san sang!")


## Bước 9 — Ham Enrich: lay SDT, Email, Nam TL
> Co checkpoint — neu bi ngat → chay lai tu cho dung


In [ ]:
def enrich_records(records, batch_size=50):
    # Load checkpoint
    enriched = set()
    if os.path.exists(CKPT_PATH):
        df_cp    = pd.read_csv(CKPT_PATH, dtype=str).fillna("")
        enriched = set(df_cp["ma_so_thue"].tolist())
        ckpt_map = df_cp.set_index("ma_so_thue").to_dict("index")
        for r in records:
            if r["ma_so_thue"] in ckpt_map:
                for k in ["so_dien_thoai","email","nam_thanh_lap","nganh_nghe"]:
                    v = ckpt_map[r["ma_so_thue"]].get(k, "")
                    if v:
                        r[k] = v
        log.info(f"Checkpoint: {len(enriched)} records da enrich")

    total = len(records)

    for i, r in enumerate(records):
        if r["ma_so_thue"] in enriched:
            continue

        href       = r.get("detail_href", "")
        detail_url = build_detail_url(href) if href else ""

        if detail_url:
            html = fetch_html(detail_url)
            if html:
                info = parse_detail(html)
                r.update(info)
        else:
            log.warning(f"Khong co href cho MST {r['ma_so_thue']}")

        done = i + 1
        if done % 10 == 0 or done == total:
            sdt_ok = sum(1 for x in records[:done] if x.get("so_dien_thoai"))
            log.info(f"  Enrich: {done}/{total} ({done/total*100:.0f}%) | Co SDT: {sdt_ok}")

        if done % batch_size == 0 or done == total:
            pd.DataFrame(records).to_csv(CKPT_PATH, index=False, encoding="utf-8-sig")

        time.sleep(random.uniform(1.5, 3.0))

    log.info(f"Enrich xong: {total} records")
    return records

print("enrich_records() san sang!")


## Bước 10 — Luu CSV + Excel (merge + dedup)

In [ ]:
def save_and_merge(records):
    COLS = [
        "ma_so_thue","ten_cong_ty","nam_thanh_lap",
        "so_dien_thoai","email","dia_chi","nganh_nghe",
        "tinh_thanh","crawled_at"
    ]
    df_moi = pd.DataFrame(records)
    for c in COLS:
        if c not in df_moi.columns:
            df_moi[c] = ""

    # Merge voi file cu
    if os.path.exists(CSV_PATH):
        df_cu = pd.read_csv(CSV_PATH, dtype=str).fillna("")
        df    = pd.concat([df_cu, df_moi[COLS]], ignore_index=True)
        print(f"Merge: {len(df_cu):,} (cu) + {len(df_moi):,} (moi) = {len(df):,}")
    else:
        df = df_moi[COLS]
        print(f"File moi: {len(df):,} records")

    # Dedup — giu record moi nhat
    before = len(df)
    df     = df.drop_duplicates(subset=["ma_so_thue"], keep="last").reset_index(drop=True)
    print(f"Dedup: {before:,} → {len(df):,} (bo {before-len(df):,} trung)")

    # Luu CSV
    df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

    # Luu Excel dep
    with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Doanh Nghiep")
        ws = writer.sheets["Doanh Nghiep"]
        col_widths = {
            "A":15,"B":45,"C":8,"D":13,"E":30,
            "F":55,"G":30,"H":10,"I":18
        }
        for col_letter, width in col_widths.items():
            ws.column_dimensions[col_letter].width = width

    print(f"\nDa luu:")
    print(f"  CSV  → {CSV_PATH}")
    print(f"  Excel→ {EXCEL_PATH}")

    # Bao cao chat luong 7 truong
    total = len(df)
    print(f"\nChat luong {total:,} records:")
    for col, label in [
        ("ma_so_thue","MST"),("ten_cong_ty","Ten CT"),
        ("dia_chi","Dia chi"),("nam_thanh_lap","Nam TL"),
        ("so_dien_thoai","SDT"),("email","Email"),("nganh_nghe","Nganh nghe")
    ]:
        n = df[col].str.strip().str.len().gt(0).sum()
        print(f"  {label:<14}: {n:>6,} ({n/total*100:.0f}%)")

    return df

print("save_and_merge() san sang!")


## Bước 11 — CHAY TOAN BO
> **Test:** `TARGET = 50`, `CO_ENRICH = True`  
> **Production:** `TARGET = 50000`, chay qua dem


In [ ]:
# ── CAU HINH ─────────────────────────────────────────────
CHAY_HA_NOI = True
CHAY_HCM    = True
TARGET      = 50      # so records moi tinh | test: 50 | full: 50000
CO_ENRICH   = True    # True = lay du SDT/Email/Nam TL

# ── PHASE 1: Crawl listing ────────────────────────────────
tat_ca = []

if CHAY_HA_NOI:
    print("="*50)
    print("  CRAWL LISTING HA NOI")
    print("="*50)
    hn = crawl_listing("ha-noi", "Ha Noi", max_records=TARGET)
    tat_ca.extend(hn)
    print(f"  → {len(hn)} records\n")

if CHAY_HCM:
    print("="*50)
    print("  CRAWL LISTING TP.HCM")
    print("="*50)
    hcm = crawl_listing("ho-chi-minh", "TP.HCM", max_records=TARGET)
    tat_ca.extend(hcm)
    print(f"  → {len(hcm)} records\n")

print(f"TONG LISTING: {len(tat_ca)} records")

# ── PHASE 2: Enrich ───────────────────────────────────────
if CO_ENRICH and tat_ca:
    print()
    print("="*50)
    print(f"  ENRICH {len(tat_ca)} records (~{len(tat_ca)*2.5/60:.0f} phut)")
    print("="*50)
    tat_ca = enrich_records(tat_ca, batch_size=50)

# ── PHASE 3: Luu ──────────────────────────────────────────
if tat_ca:
    df_final = save_and_merge(tat_ca)
    print(f"\nHOAN THANH! {len(df_final):,} records trong file")


## Bước 12 — Dong browser

In [ ]:
try:
    browser.close()
    _pw.stop()
    print("Browser da dong!")
except Exception:
    print("Browser da dong truoc do")


## Bước 13 — Xem ket qua

In [ ]:
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH, dtype=str).fillna("")
    total = len(df)
    print(f"File: {CSV_PATH}")
    print(f"Tong: {total:,} records")
    print()
    print("Phan bo tinh thanh:")
    print(df["tinh_thanh"].value_counts().to_string())
    print()
    print("Chat luong 7 truong:")
    for col, label in [
        ("ma_so_thue","MST"),("ten_cong_ty","Ten CT"),
        ("dia_chi","Dia chi"),("nam_thanh_lap","Nam TL"),
        ("so_dien_thoai","SDT"),("email","Email"),("nganh_nghe","Nganh nghe")
    ]:
        n = df[col].str.strip().str.len().gt(0).sum()
        print(f"  {label:<14}: {n:>6,} ({n/total*100:.0f}%)")
    print()
    print("5 records mau:")
    print(df[["ma_so_thue","ten_cong_ty","so_dien_thoai","email","nam_thanh_lap"]].head(5).to_string(index=False))
else:
    print("Chua co file — chay Buoc 11 truoc!")
